# 🥭 Mango Anthracnose Disease Detection Model Training

This notebook trains a deep learning model to detect anthracnose disease in mango fruits and leaves.

## Overview
- **Dataset:** 4,709 unique images (1,899 healthy + 2,810 anthracnose)
- **Model:** ResNet50 with transfer learning
- **Task:** Binary classification (Healthy vs Anthracnose)
- **Framework:** PyTorch

## Dataset Composition
- **Fruits:** 486 images (249 healthy, 237 anthracnose)
- **Leaves:** 4,223 images (1,650 healthy, 2,573 anthracnose)
- **Duplicates Removed:** 1,702 (26.5% deduplication rate)

## Dataset Structure
```
anthracnose/
├── train/ (3,295 images: 1,329 healthy, 1,966 anthracnose)
├── val/   (705 images: 284 healthy, 421 anthracnose)
└── test/  (709 images: 286 healthy, 423 anthracnose)
```

## 📦 1. Install Required Packages

In [ ]:
# Install required packages (uncomment if needed on Kaggle)
# !pip install torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu118
# !pip install timm albumentations scikit-learn matplotlib seaborn tqdm pillow

## 📚 2. Import Libraries

In [ ]:
import os
import random
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
from tqdm.auto import tqdm
from PIL import Image
import warnings
warnings.filterwarnings('ignore')

# PyTorch imports
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms, models

# Sklearn imports
from sklearn.metrics import (
    classification_report, 
    confusion_matrix, 
    accuracy_score,
    precision_recall_fscore_support,
    roc_auc_score,
    roc_curve
)

# Set random seeds for reproducibility
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed(SEED)
torch.backends.cudnn.deterministic = True

print(f"PyTorch Version: {torch.__version__}")
print(f"CUDA Available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"CUDA Device: {torch.cuda.get_device_name(0)}")

## ⚙️ 3. Configuration and Hyperparameters

In [ ]:
# Configuration
class Config:
    # Paths - UPDATE THESE BASED ON YOUR ENVIRONMENT
    # For local testing, use local path
    # For Kaggle, use Kaggle input path
    
    # LOCAL PATH (comment out when running on Kaggle)
    # DATA_DIR = Path('../data/unified/anthracnose')  # Local path
    # OUTPUT_DIR = Path('../output')  # Local output
    
    # KAGGLE PATH (uncomment when running on Kaggle)
    DATA_DIR = Path('/kaggle/input/anthracnose-complete-dataset/anthracnose')
    OUTPUT_DIR = Path('/kaggle/working')
    
    # Model parameters
    MODEL_NAME = 'resnet50'
    NUM_CLASSES = 2  # Binary: Healthy vs Anthracnose
    IMAGE_SIZE = 224
    
    # Training parameters
    BATCH_SIZE = 32
    NUM_EPOCHS = 50
    LEARNING_RATE = 0.001
    WEIGHT_DECAY = 1e-4
    
    # Data parameters
    NUM_WORKERS = 0  # Set to 0 to avoid multiprocessing errors
    PIN_MEMORY = True
    
    # Training parameters
    EARLY_STOPPING_PATIENCE = 10
    REDUCE_LR_PATIENCE = 5
    
    # Device
    DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

config = Config()
config.OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print(f"Dataset Directory: {config.DATA_DIR.absolute()}")
print(f"Dataset Exists: {config.DATA_DIR.exists()}")
print(f"Output Directory: {config.OUTPUT_DIR.absolute()}")
print(f"Device: {config.DEVICE}")
print(f"Batch Size: {config.BATCH_SIZE}")
print(f"Image Size: {config.IMAGE_SIZE}")
print(f"Learning Rate: {config.LEARNING_RATE}")
print(f"Number of Epochs: {config.NUM_EPOCHS}")

## 📁 4. Dataset Class Definition

In [ ]:
class AnthracnoseDataset(Dataset):
    """Custom Dataset for Anthracnose Detection"""
    
    def __init__(self, data_dir, split='train', transform=None):
        """
        Args:
            data_dir: Path to dataset root (contains healthy/ and anthracnose/ folders)
            split: 'train', 'val', or 'test'
            transform: Data augmentation/transformation pipeline
        """
        self.data_dir = Path(data_dir)
        self.split = split
        self.transform = transform
        
        # Class mapping
        self.class_to_idx = {'healthy': 0, 'anthracnose': 1}
        self.idx_to_class = {0: 'healthy', 1: 'anthracnose'}
        
        # Load image paths and labels
        self.images = []
        self.labels = []
        
        # Load healthy images
        healthy_dir = self.data_dir / split / 'healthy'
        if healthy_dir.exists():
            for img_path in healthy_dir.glob('*.jpg'):
                self.images.append(img_path)
                self.labels.append(0)  # Healthy
        
        # Load anthracnose images
        anthracnose_dir = self.data_dir / split / 'anthracnose'
        if anthracnose_dir.exists():
            for img_path in anthracnose_dir.glob('*.jpg'):
                self.images.append(img_path)
                self.labels.append(1)  # Anthracnose
        
        print(f"{split.upper()} Dataset: {len(self.images)} images")
        print(f"  - Healthy: {sum(1 for l in self.labels if l == 0)}")
        print(f"  - Anthracnose: {sum(1 for l in self.labels if l == 1)}")
    
    def __len__(self):
        return len(self.images)
    
    def __getitem__(self, idx):
        # Load image
        img_path = self.images[idx]
        image = Image.open(img_path).convert('RGB')
        label = self.labels[idx]
        
        # Apply transformations
        if self.transform:
            image = self.transform(image)
        
        return image, label

## 🔄 5. Data Augmentation and Transformations

In [ ]:
# Training transformations (with augmentation)
train_transform = transforms.Compose([
    transforms.Resize((256, 256)),
    transforms.RandomCrop(config.IMAGE_SIZE),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomVerticalFlip(p=0.3),
    transforms.RandomRotation(degrees=30),
    transforms.ColorJitter(
        brightness=0.2,
        contrast=0.2,
        saturation=0.2,
        hue=0.1
    ),
    transforms.RandomAffine(
        degrees=0,
        translate=(0.1, 0.1),
        scale=(0.9, 1.1)
    ),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])

# Validation/Test transformations (no augmentation)
val_transform = transforms.Compose([
    transforms.Resize((config.IMAGE_SIZE, config.IMAGE_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])

print("✅ Transformations defined")

## 📊 6. Create Data Loaders

In [ ]:
# Create datasets
train_dataset = AnthracnoseDataset(
    config.DATA_DIR,
    split='train',
    transform=train_transform
)

val_dataset = AnthracnoseDataset(
    config.DATA_DIR,
    split='val',
    transform=val_transform
)

test_dataset = AnthracnoseDataset(
    config.DATA_DIR,
    split='test',
    transform=val_transform
)

# Create data loaders
train_loader = DataLoader(
    train_dataset,
    batch_size=config.BATCH_SIZE,
    shuffle=True,
    num_workers=config.NUM_WORKERS,
    pin_memory=config.PIN_MEMORY
)

val_loader = DataLoader(
    val_dataset,
    batch_size=config.BATCH_SIZE,
    shuffle=False,
    num_workers=config.NUM_WORKERS,
    pin_memory=config.PIN_MEMORY
)

test_loader = DataLoader(
    test_dataset,
    batch_size=config.BATCH_SIZE,
    shuffle=False,
    num_workers=config.NUM_WORKERS,
    pin_memory=config.PIN_MEMORY
)

print(f"\n✅ Data loaders created")
print(f"Train batches: {len(train_loader)}")
print(f"Val batches: {len(val_loader)}")
print(f"Test batches: {len(test_loader)}")

## 🖼️ 7. Visualize Sample Images

In [ ]:
def show_batch(loader, n_images=8):
    """Display a batch of images"""
    # Get a batch
    images, labels = next(iter(loader))
    
    # Denormalize images
    mean = torch.tensor([0.485, 0.456, 0.406]).view(3, 1, 1)
    std = torch.tensor([0.229, 0.224, 0.225]).view(3, 1, 1)
    images = images * std + mean
    
    # Plot
    fig, axes = plt.subplots(2, 4, figsize=(16, 8))
    axes = axes.flatten()
    
    for i in range(min(n_images, len(images))):
        img = images[i].permute(1, 2, 0).cpu().numpy()
        img = np.clip(img, 0, 1)
        
        label_name = 'Healthy' if labels[i] == 0 else 'Anthracnose'
        color = 'green' if labels[i] == 0 else 'red'
        
        axes[i].imshow(img)
        axes[i].set_title(label_name, color=color, fontweight='bold', fontsize=12)
        axes[i].axis('off')
    
    plt.tight_layout()
    plt.savefig(config.OUTPUT_DIR / 'sample_images.png', dpi=150, bbox_inches='tight')
    plt.show()

show_batch(train_loader)

## 🏗️ 8. Define Model Architecture

In [ ]:
def create_model(model_name='resnet50', num_classes=2, pretrained=True):
    """Create a model with transfer learning"""
    
    if model_name == 'resnet50':
        model = models.resnet50(pretrained=pretrained)
        num_features = model.fc.in_features
        model.fc = nn.Linear(num_features, num_classes)
    
    elif model_name == 'resnet34':
        model = models.resnet34(pretrained=pretrained)
        num_features = model.fc.in_features
        model.fc = nn.Linear(num_features, num_classes)
    
    elif model_name == 'efficientnet_b0':
        model = models.efficientnet_b0(pretrained=pretrained)
        num_features = model.classifier[1].in_features
        model.classifier[1] = nn.Linear(num_features, num_classes)
    
    elif model_name == 'mobilenet_v2':
        model = models.mobilenet_v2(pretrained=pretrained)
        num_features = model.classifier[1].in_features
        model.classifier[1] = nn.Linear(num_features, num_classes)
    
    else:
        raise ValueError(f"Unknown model: {model_name}")
    
    return model

# Create model
model = create_model(
    model_name=config.MODEL_NAME,
    num_classes=config.NUM_CLASSES,
    pretrained=True
)

model = model.to(config.DEVICE)

# Count parameters
total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)

print(f"\n✅ Model: {config.MODEL_NAME}")
print(f"Total parameters: {total_params:,}")
print(f"Trainable parameters: {trainable_params:,}")

## 🎯 9. Define Loss Function and Optimizer

In [ ]:
# Loss function
criterion = nn.CrossEntropyLoss()

# Optimizer
optimizer = optim.Adam(
    model.parameters(),
    lr=config.LEARNING_RATE,
    weight_decay=config.WEIGHT_DECAY
)

# Learning rate scheduler
scheduler = optim.lr_scheduler.ReduceLROnPlateau(
    optimizer,
    mode='min',
    factor=0.5,
    patience=config.REDUCE_LR_PATIENCE,
    verbose=True
)

print("✅ Loss function, optimizer, and scheduler defined")

## 🏋️ 10. Training and Validation Functions

In [ ]:
def train_epoch(model, loader, criterion, optimizer, device):
    """Train for one epoch"""
    model.train()
    running_loss = 0.0
    correct = 0
    total = 0
    
    pbar = tqdm(loader, desc='Training')
    for images, labels in pbar:
        images = images.to(device)
        labels = labels.to(device)
        
        # Forward pass
        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        
        # Backward pass
        loss.backward()
        optimizer.step()
        
        # Statistics
        running_loss += loss.item() * images.size(0)
        _, predicted = outputs.max(1)
        total += labels.size(0)
        correct += predicted.eq(labels).sum().item()
        
        # Update progress bar
        pbar.set_postfix({
            'loss': f'{loss.item():.4f}',
            'acc': f'{100.*correct/total:.2f}%'
        })
    
    epoch_loss = running_loss / len(loader.dataset)
    epoch_acc = 100. * correct / total
    
    return epoch_loss, epoch_acc


def validate_epoch(model, loader, criterion, device):
    """Validate for one epoch"""
    model.eval()
    running_loss = 0.0
    correct = 0
    total = 0
    
    all_preds = []
    all_labels = []
    
    with torch.no_grad():
        pbar = tqdm(loader, desc='Validation')
        for images, labels in pbar:
            images = images.to(device)
            labels = labels.to(device)
            
            # Forward pass
            outputs = model(images)
            loss = criterion(outputs, labels)
            
            # Statistics
            running_loss += loss.item() * images.size(0)
            _, predicted = outputs.max(1)
            total += labels.size(0)
            correct += predicted.eq(labels).sum().item()
            
            all_preds.extend(predicted.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())
            
            # Update progress bar
            pbar.set_postfix({
                'loss': f'{loss.item():.4f}',
                'acc': f'{100.*correct/total:.2f}%'
            })
    
    epoch_loss = running_loss / len(loader.dataset)
    epoch_acc = 100. * correct / total
    
    return epoch_loss, epoch_acc, all_preds, all_labels

print("✅ Training and validation functions defined")

## 🚀 11. Train the Model

In [ ]:
# Training history
history = {
    'train_loss': [],
    'train_acc': [],
    'val_loss': [],
    'val_acc': []
}

# Best model tracking
best_val_acc = 0.0
best_model_path = config.OUTPUT_DIR / 'best_model.pth'
patience_counter = 0

print("\n" + "="*70)
print("🚀 STARTING TRAINING")
print("="*70)

for epoch in range(config.NUM_EPOCHS):
    print(f"\n📅 Epoch {epoch+1}/{config.NUM_EPOCHS}")
    print("-" * 70)
    
    # Train
    train_loss, train_acc = train_epoch(
        model, train_loader, criterion, optimizer, config.DEVICE
    )
    
    # Validate
    val_loss, val_acc, val_preds, val_labels = validate_epoch(
        model, val_loader, criterion, config.DEVICE
    )
    
    # Update scheduler
    scheduler.step(val_loss)
    
    # Save history
    history['train_loss'].append(train_loss)
    history['train_acc'].append(train_acc)
    history['val_loss'].append(val_loss)
    history['val_acc'].append(val_acc)
    
    # Print epoch summary
    print(f"\n📊 Epoch {epoch+1} Summary:")
    print(f"   Train Loss: {train_loss:.4f} | Train Acc: {train_acc:.2f}%")
    print(f"   Val Loss:   {val_loss:.4f} | Val Acc:   {val_acc:.2f}%")
    print(f"   LR: {optimizer.param_groups[0]['lr']:.6f}")
    
    # Save best model
    if val_acc > best_val_acc:
        best_val_acc = val_acc
        torch.save({
            'epoch': epoch,
            'model_state_dict': model.state_dict(),
            'optimizer_state_dict': optimizer.state_dict(),
            'val_acc': val_acc,
            'val_loss': val_loss,
        }, best_model_path)
        print(f"   ✅ New best model saved! (Val Acc: {val_acc:.2f}%)")
        patience_counter = 0
    else:
        patience_counter += 1
    
    # Early stopping
    if patience_counter >= config.EARLY_STOPPING_PATIENCE:
        print(f"\n⏹️ Early stopping triggered after {epoch+1} epochs")
        break
    
    # Save checkpoint every 10 epochs
    if (epoch + 1) % 10 == 0:
        checkpoint_path = config.OUTPUT_DIR / f'checkpoint_epoch_{epoch+1}.pth'
        torch.save({
            'epoch': epoch,
            'model_state_dict': model.state_dict(),
            'optimizer_state_dict': optimizer.state_dict(),
        }, checkpoint_path)

print("\n" + "="*70)
print("✨ TRAINING COMPLETE!")
print(f"🏆 Best Validation Accuracy: {best_val_acc:.2f}%")
print("="*70)

## 📈 12. Plot Training History

In [ ]:
def plot_training_history(history):
    """Plot training and validation metrics"""
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(15, 5))
    
    epochs = range(1, len(history['train_loss']) + 1)
    
    # Plot loss
    ax1.plot(epochs, history['train_loss'], 'b-', label='Train Loss', linewidth=2)
    ax1.plot(epochs, history['val_loss'], 'r-', label='Val Loss', linewidth=2)
    ax1.set_xlabel('Epoch', fontsize=12)
    ax1.set_ylabel('Loss', fontsize=12)
    ax1.set_title('Training and Validation Loss', fontsize=14, fontweight='bold')
    ax1.legend(fontsize=11)
    ax1.grid(True, alpha=0.3)
    
    # Plot accuracy
    ax2.plot(epochs, history['train_acc'], 'b-', label='Train Acc', linewidth=2)
    ax2.plot(epochs, history['val_acc'], 'r-', label='Val Acc', linewidth=2)
    ax2.set_xlabel('Epoch', fontsize=12)
    ax2.set_ylabel('Accuracy (%)', fontsize=12)
    ax2.set_title('Training and Validation Accuracy', fontsize=14, fontweight='bold')
    ax2.legend(fontsize=11)
    ax2.grid(True, alpha=0.3)
    
    plt.tight_layout()
    plt.savefig(config.OUTPUT_DIR / 'training_history.png', dpi=300, bbox_inches='tight')
    plt.show()

plot_training_history(history)

## 🧪 13. Evaluate on Test Set

In [ ]:
# Load best model
checkpoint = torch.load(best_model_path)
model.load_state_dict(checkpoint['model_state_dict'])
print(f"✅ Loaded best model from epoch {checkpoint['epoch']+1}")
print(f"   Best Val Acc: {checkpoint['val_acc']:.2f}%")

# Evaluate on test set
print("\n" + "="*70)
print("🧪 EVALUATING ON TEST SET")
print("="*70 + "\n")

test_loss, test_acc, test_preds, test_labels = validate_epoch(
    model, test_loader, criterion, config.DEVICE
)

print(f"\n📊 Test Results:")
print(f"   Test Loss: {test_loss:.4f}")
print(f"   Test Accuracy: {test_acc:.2f}%")

## 📊 14. Classification Report and Metrics

In [ ]:
# Classification report
class_names = ['Healthy', 'Anthracnose']
print("\n" + "="*70)
print("📊 CLASSIFICATION REPORT")
print("="*70 + "\n")
print(classification_report(test_labels, test_preds, target_names=class_names))

# Calculate additional metrics
precision, recall, f1, support = precision_recall_fscore_support(
    test_labels, test_preds, average='weighted'
)

print("\n" + "="*70)
print("📈 OVERALL METRICS")
print("="*70)
print(f"Accuracy:  {test_acc:.2f}%")
print(f"Precision: {precision*100:.2f}%")
print(f"Recall:    {recall*100:.2f}%")
print(f"F1-Score:  {f1*100:.2f}%")
print("="*70)

## 🎨 15. Confusion Matrix Visualization

In [ ]:
def plot_confusion_matrix(y_true, y_pred, class_names):
    """Plot confusion matrix"""
    cm = confusion_matrix(y_true, y_pred)
    
    plt.figure(figsize=(10, 8))
    sns.heatmap(
        cm,
        annot=True,
        fmt='d',
        cmap='Blues',
        xticklabels=class_names,
        yticklabels=class_names,
        cbar_kws={'label': 'Count'},
        annot_kws={'size': 14, 'weight': 'bold'}
    )
    
    plt.ylabel('True Label', fontsize=12, fontweight='bold')
    plt.xlabel('Predicted Label', fontsize=12, fontweight='bold')
    plt.title('Confusion Matrix - Test Set', fontsize=14, fontweight='bold', pad=20)
    
    # Add percentages
    for i in range(len(class_names)):
        for j in range(len(class_names)):
            percentage = cm[i, j] / cm[i].sum() * 100
            plt.text(
                j + 0.5, i + 0.7,
                f'({percentage:.1f}%)',
                ha='center', va='center',
                fontsize=10, color='gray'
            )
    
    plt.tight_layout()
    plt.savefig(config.OUTPUT_DIR / 'confusion_matrix.png', dpi=300, bbox_inches='tight')
    plt.show()

plot_confusion_matrix(test_labels, test_preds, class_names)

## 💾 16. Save Final Model and Results

In [ ]:
# Save final model
final_model_path = config.OUTPUT_DIR / 'anthracnose_detection_model.pth'
torch.save({
    'model_state_dict': model.state_dict(),
    'model_name': config.MODEL_NAME,
    'num_classes': config.NUM_CLASSES,
    'image_size': config.IMAGE_SIZE,
    'class_names': class_names,
    'test_accuracy': test_acc,
    'best_val_accuracy': best_val_acc,
}, final_model_path)

print(f"\n✅ Final model saved to: {final_model_path}")

# Save training history
history_df = pd.DataFrame(history)
history_df.to_csv(config.OUTPUT_DIR / 'training_history.csv', index=False)
print(f"✅ Training history saved")

# Save test results
results = {
    'test_accuracy': test_acc,
    'test_loss': test_loss,
    'best_val_accuracy': best_val_acc,
    'precision': precision * 100,
    'recall': recall * 100,
    'f1_score': f1 * 100,
}

results_df = pd.DataFrame([results])
results_df.to_csv(config.OUTPUT_DIR / 'test_results.csv', index=False)
print(f"✅ Test results saved")

print("\n" + "="*70)
print("🎉 ALL DONE!")
print("="*70)
print(f"\n📁 All outputs saved to: {config.OUTPUT_DIR}")
print("\nGenerated files:")
print("  📊 training_history.png      - Training curves")
print("  🎨 confusion_matrix.png      - Confusion matrix")
print("  🖼️  sample_images.png         - Sample images")
print("  💾 best_model.pth            - Best checkpoint")
print("  💾 anthracnose_detection_model.pth - Final model")
print("  📄 training_history.csv      - Training metrics")
print("  📄 test_results.csv          - Test metrics")
print("\n" + "="*70)

## 🔮 17. Inference Function (Optional)

In [ ]:
def predict_image(model, image_path, transform, device, class_names):
    """Predict a single image"""
    model.eval()
    
    # Load and transform image
    image = Image.open(image_path).convert('RGB')
    image_tensor = transform(image).unsqueeze(0).to(device)
    
    # Predict
    with torch.no_grad():
        outputs = model(image_tensor)
        probabilities = torch.nn.functional.softmax(outputs, dim=1)
        confidence, predicted = torch.max(probabilities, 1)
    
    pred_class = class_names[predicted.item()]
    confidence = confidence.item() * 100
    
    # Visualize
    plt.figure(figsize=(10, 6))
    plt.imshow(image)
    plt.axis('off')
    
    color = 'green' if pred_class == 'Healthy' else 'red'
    plt.title(
        f"Prediction: {pred_class}\nConfidence: {confidence:.2f}%",
        fontsize=14,
        fontweight='bold',
        color=color,
        pad=20
    )
    plt.tight_layout()
    plt.show()
    
    return pred_class, confidence

print("✅ Inference function defined")
print("\nUsage:")
print("predict_image(model, 'path/to/image.jpg', val_transform, config.DEVICE, class_names)")

## 📝 18. Summary and Next Steps

### 🎉 Training Complete!

**Model Performance:**
- Test Accuracy: Check the output above
- Model Architecture: ResNet50 with transfer learning
- Total Parameters: ~25M

### 📦 Deliverables

1. **Trained Model:** `anthracnose_detection_model.pth`
2. **Best Checkpoint:** `best_model.pth`
3. **Training Curves:** `training_history.png`
4. **Confusion Matrix:** `confusion_matrix.png`
5. **Metrics CSV:** `test_results.csv`

### 🚀 Next Steps

1. **Download the model** from Kaggle output directory
2. **Integrate into API** (FRESH ML backend)
3. **Deploy to production** server
4. **Test with real images** from field
5. **Monitor performance** and collect feedback

### 🔧 Model Optimization

If you need better performance:
- Try different architectures (EfficientNet, Vision Transformer)
- Increase training epochs
- Use more data augmentation
- Experiment with different learning rates
- Use ensemble methods

### 📚 Resources

- **Dataset:** `/kaggle/input/anthracnose-dataset`
- **Model Docs:** See `QUICKSTART.md` in dataset folder
- **Training Script:** This notebook

---

**Created for:** FRESH ML - Module 2 (Disease Detection)  
**Date:** October 2025  
**Model:** Anthracnose Detection in Mangoes